# 🧬 Day 2 — AI 두 개를 레고처럼 합치기 (왕초보 버전)

## 오늘 배울 것, 딱 한 문장
> **재학습(새로 가르치기) 없이, 이미 있는 AI 두 개를 섞어서 '둘 다 잘하는' 새 AI를 만든다.**

이게 바로 어제 강의의 David Ha(사카나 CEO)의 대표 기술 — **모델 머징(Evolutionary Merge)** 입니다.

## 🤝 약속 (어제와 같아요)
1. **코드는 안 읽어도 됩니다.** ▶만 누르면 한글 해설이 같이 나옵니다.
2. 어려운 말은 옆에 쉬운 말로 풀어놨어요.
3. 그림·표만 보고 지나가도 오늘 목표 달성입니다.

## 준비
- **런타임 → 런타임 유형 변경 → T4 GPU → 저장**
- 위에서부터 순서대로 ▶

## 오늘의 완료 조건
- [ ] '섞는 비율'을 바꾸면 성적이 달라지는 표 1장 보기
- [ ] 우승한 비율로 만든 새 AI가 실제로 답하는 것 보기
- [ ] 마지막 빈칸 3개 채우기

---
## 📖 1교시 — '모델 합치기'가 뭔가요? (읽기만, 3분)

### 비유: 요리 레시피 섞기 🍲
- AI 하나는 **한식 잘하는 요리사**, 다른 하나는 **양식 잘하는 요리사** 라고 해봐요.
- 둘을 **7:3, 5:5, 3:7** 처럼 비율을 정해 **섞으면** → 한식도 양식도 그럭저럭 하는 **퓨전 요리사**가 태어납니다.
- 놀라운 점: **새로 요리를 안 가르쳐도** 됩니다. 이미 배운 둘을 *섞기만* 하는 거예요. (그래서 GPU도 거의 안 듦 = 0원에 가까움)

### 오늘 우리가 할 진짜 실험
1. AI 두 개를 데려온다 (하나는 '지시 잘 따르는' AI, 하나는 '수학/코딩 쪽' AI)
2. **섞는 비율을 여러 개** 만든다 (2:8, 3.5:6.5, 5:5, 6.5:3.5, 8:2)
3. 각 비율로 만든 AI에게 **똑같은 시험**을 보게 해서 점수를 매긴다
4. **1등 비율만 남긴다** ← 이게 바로 사카나의 '진화(자연선택)'를 손으로 하는 것!

### 어려운 단어 미리 풀기
| 단어 | 쉬운 말 |
| --- | --- |
| 머징(merge) | AI 섞기 |
| 가중치(weight) | AI가 배운 내용이 담긴 '숫자 뭉치' |
| 진화 병합 | 여러 비율을 만들어 시험 보고, 성적 좋은 것만 남기기 |
| SLERP | 두 숫자 뭉치를 '매끄럽게' 섞는 방법 (이름만 알면 됨) |

> ⚠️ **딱 하나 규칙:** 섞는 두 AI는 **같은 집안(같은 크기·같은 계열)** 이어야 해요. 한식 요리사끼리는 섞이지만, 요리사랑 목수는 못 섞어요! 오늘은 안전하게 같은 집안 둘로 갑니다.

이제 해봅시다 👇

## ① 도구 설치 (▶, 1~2분)

In [ ]:
# ▶만 누르세요. (AI 섞기 도구 + AI 실행 도구 설치)
%pip install -q mergekit transformers accelerate torch
print('✅ 설치 끝!')

## ② 섞을 AI 두 개 정하기 + 시험 문제 만들기 (▶)

In [ ]:
# ▶만 누르세요. (섞을 재료 AI 두 개와, 우승자를 가릴 시험지를 정합니다)

# 같은 집안(Qwen2.5-0.5B)의 형제 둘 — 안전하게 섞이는 조합
MODEL_A = 'Qwen/Qwen2.5-0.5B-Instruct'   # 형: 지시를 잘 따르고 말을 잘함
MODEL_B = 'Qwen/Qwen2.5-Coder-0.5B'      # 동생: 코딩/논리 쪽에 강함

# 시험지: 두 능력(말하기 + 논리)을 섞어 물어봄. 채점은 '정답 단어가 들어있나'로 간단히.
quiz = [
    {'q': '3 곱하기 4는 얼마야? 숫자만 답해.', 'answer': '12'},
    {'q': '10 빼기 7은?', 'answer': '3'},
    {'q': '대한민국의 수도는 어디야?', 'answer': '서울'},
    {'q': '"사과"를 영어로?', 'answer': 'apple'},
    {'q': '2 더하기 2 곱하기 2는? 숫자만.', 'answer': '6'},
]
print(f'재료 AI 2개 준비 완료:\n  형: {MODEL_A}\n  동생: {MODEL_B}')
print(f'\n우승자를 가릴 시험 문제 {len(quiz)}개도 준비됐어요.')
print('(수학+상식+영어를 섞어서 냈어요 — 두 AI의 능력이 모두 필요하게)')

## ③ 여러 비율로 섞고, 시험 보게 하기 🏁 (▶, 여기가 오늘의 핵심! 5~10분)

5가지 비율(20%, 35%, 50%, 65%, 80%)로 AI를 섞어서 각각 시험을 보게 합니다.
비율 하나마다 '섞기 → 시험'이 돌아가서 조금 걸려요. 커피 한 잔 ☕

In [ ]:
# ▶만 누르세요. (5가지 비율로 섞고 → 각각 시험 → 점수표. 조금 걸립니다)
import torch, gc, os, tempfile
from transformers import AutoModelForCausalLM, AutoTokenizer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
tok = AutoTokenizer.from_pretrained(MODEL_A)

# 두 AI의 '숫자 뭉치'를 미리 메모리에 올려둠 (매번 안 불러오려고)
print('재료 AI 2개를 메모리에 올리는 중...')
base_a = AutoModelForCausalLM.from_pretrained(MODEL_A, torch_dtype=torch.float32)
base_b = AutoModelForCausalLM.from_pretrained(MODEL_B, torch_dtype=torch.float32)
sd_a, sd_b = base_a.state_dict(), base_b.state_dict()
shared = [k for k in sd_a if k in sd_b and sd_a[k].shape == sd_b[k].shape]

def make_mixed_model(ratio_b):
    """동생(B)을 ratio_b 비율로 섞은 새 AI를 만든다 (SLERP의 쉬운 사촌 = 선형 섞기)."""
    merged = AutoModelForCausalLM.from_pretrained(MODEL_A, torch_dtype=torch.float32)
    msd = merged.state_dict()
    for k in shared:
        msd[k] = (1 - ratio_b) * sd_a[k] + ratio_b * sd_b[k]   # ← '섞기'의 전부
    merged.load_state_dict(msd)
    return merged.to(device).eval()

def take_exam(m):
    """시험 5문제를 풀리고 맞은 개수를 센다."""
    score = 0
    for item in quiz:
        msgs = [{'role': 'user', 'content': item['q']}]
        prompt = tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
        ids = tok(prompt, return_tensors='pt').to(device)
        out = m.generate(**ids, max_new_tokens=30, do_sample=False, pad_token_id=tok.eos_token_id)
        ans = tok.decode(out[0][ids['input_ids'].shape[1]:], skip_special_tokens=True)
        if item['answer'].lower() in ans.lower():
            score += 1
    return score

ratios = [0.20, 0.35, 0.50, 0.65, 0.80]
results = []
print('\n비율별 섞기 + 시험 시작!\n')
for r in ratios:
    m = make_mixed_model(r)
    s = take_exam(m)
    results.append((r, s))
    bar = '🟩' * s + '⬜' * (len(quiz) - s)
    print(f"  형{int((1-r)*100)}% : 동생{int(r*100)}%  →  {bar}  {s}/{len(quiz)}점")
    del m; gc.collect(); torch.cuda.empty_cache() if device == 'cuda' else None

winner = max(results, key=lambda x: x[1])
print(f"\n🏆 우승 비율: 형{int((1-winner[0])*100)}% : 동생{int(winner[0]*100)}%  ({winner[1]}/{len(quiz)}점)")
print('   → 이게 바로 사카나의 "진화": 여러 개 만들어 시험 보고, 제일 잘한 것만 살아남기!')

✅ **여기까지가 오늘의 하이라이트!** 비율(레시피)에 따라 점수가 달라지는 게 보이죠? 사람이 "이 비율이 좋겠지" 찍는 게 아니라, **여러 개 만들어 시험으로 고르는 것** — 이게 진화 병합의 핵심입니다.

## ④ 우승한 비율로 만든 AI와 실제로 대화해보기 (▶)

In [ ]:
# ▶만 누르세요. (우승 비율로 새 AI를 만들어 직접 답하게)
champion = make_mixed_model(winner[0])

def ask(q):
    msgs = [{'role': 'user', 'content': q}]
    prompt = tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
    ids = tok(prompt, return_tensors='pt').to(device)
    out = champion.generate(**ids, max_new_tokens=60, do_sample=False, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][ids['input_ids'].shape[1]:], skip_special_tokens=True).strip()

for q in ['안녕! 너를 한 문장으로 소개해줘.', '15 곱하기 3은 얼마야?', '봄에 대한 짧은 한 줄을 써줘.']:
    print(f'Q: {q}')
    print(f'A: {ask(q)}\n')

print('📖 보는 법: 말하기(형)와 계산(동생)이 한 AI 안에 섞였는지 구경하세요.')
print('   꼬마 AI라 완벽하진 않아요 — "둘을 안 가르치고 섞기만 했다"가 핵심 관전 포인트!')

---
## 🏁 오늘의 마무리 — 빈칸 3개 채우면 졸업!

1. 모델 머징이란 이미 있는 AI를 새로 가르치지 않고 (          )서 새 AI를 만드는 것이다.
2. 사카나의 '진화'란, 여러 (          )로 만들어 (          )을 보고 제일 잘한 것만 남기는 것이다.
3. 섞는 두 AI는 반드시 같은 (          )이어야 한다. (요리사끼리는 되지만 요리사+목수는 안 됨)

<details><summary>정답 보기 (클릭)</summary>
1. 섞어(합쳐)  ·  2. 비율 / 시험(성적)  ·  3. 집안(계열·크기)
</details>

### 오늘 이해한 것과 내일의 연결
- 오늘: **재학습 없이 AI를 섞어** 새 능력을 만들었고, **시험으로 우승 레시피를 골랐습니다** (사카나의 진화).
- 지금까지: Day1(뇌 구경) → Day2(뇌 합치기). 이제 남은 건 **파는 것!**
- 내일(Day 3): 지금까지 만든 AI들을 **'상품'으로 포장**합니다 — 이름·가격·소개 한 장. (코드 거의 없음, 워크시트로!)

> 랩노트(`ha-merge.md`)에는 ①점수표 캡처 ②우승 비율 ③빈칸 답 — 3개면 끝!

### 🤓 (궁금한 사람만) mergekit YAML은 이렇게 생겼어요
우리는 이해를 위해 손으로 섞었지만, 실무에선 `mergekit`에 이런 설정 파일을 주면 자동으로 해줍니다:
```yaml
models:
  - model: Qwen/Qwen2.5-0.5B-Instruct
  - model: Qwen/Qwen2.5-Coder-0.5B
merge_method: slerp
base_model: Qwen/Qwen2.5-0.5B-Instruct
parameters:
  t: 0.5   # ← 이 숫자가 오늘 우리가 바꿔본 '섞는 비율'
dtype: bfloat16
```